# 🛠️ Feature Engineering & Data Preprocessing
## Modular Pipeline Exploration

This notebook demonstrates the preprocessing and feature engineering steps applied to the dataset using modular functions defined in `src/`. Preprocessing includes median imputation, IQR winsorizing for outlier capping, ratio feature creation, categorical one-hot encoding, and standard scaling.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('../src'))
from data_loader import load_data
from preprocessing import handle_missing_values, cap_outliers_iqr
from feature_engineering import add_derived_features, encode_categorical, scale_features

sns.set_theme(style='whitegrid')
%matplotlib inline

### 1. Missing Value Imputation
Imputing missing numerical entries (`total_bedrooms`) using column median values.

In [ ]:
raw_df = load_data()
df = handle_missing_values(raw_df)
print('Missing values remaining:', df.isnull().sum().sum())

### 2. Outlier Winsorization via IQR Rule
Capping extreme outliers in heavy-tailed features (`total_rooms`, `total_bedrooms`, `population`, `households`, `median_income`) to improve model stability.

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include='number').columns if c != 'median_house_value']
df_capped = cap_outliers_iqr(df, numeric_cols)
print('Outliers capped successfully.')

### 3. Domain-Informed Ratio Features
Constructing 3 derived interaction terms:
- `rooms_per_household` = `total_rooms` / `households`
- `bedrooms_per_room` = `total_bedrooms` / `total_rooms`
- `population_per_household` = `population` / `households`

In [ ]:
df_feat = add_derived_features(df_capped)
df_feat[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].describe()

### 4. Categorical One-Hot Encoding
Converting `ocean_proximity` into binary indicators and sanitizing column headers for tree model compatibility.

In [ ]:
df_encoded = encode_categorical(df_feat)
print('New encoded columns:', [c for c in df_encoded.columns if c.startswith('ocean_proximity')])

### 5. Standard Scaling
Scaling numerical feature representations using `StandardScaler` fitted on training split.

In [ ]:
X = df_encoded.drop(columns=['median_house_value'])
y = df_encoded['median_house_value']
X_scaled, _, scaler = scale_features(X, X)
X_scaled.head()